In [ ]:

from sympy import *

# projection of column vector v
# onto column space of U
# with orthonormal columns

def project_to_ortho(U, v):
	x = U.T*v        # reduced
	return U*x       # projected

def gs(A):
	d, N = A.shape
	U = zeros(d, 0)
	for j in range(N):
		v = A[:, j]
		Pv = project_to_ortho(U, v)
		w = simplify(v - Pv)
		if not w.is_zero_matrix:
			u = w / sqrt(simplify((w.T*w)[0]))
			U = Matrix.hstack(U, simplify(u))
	return U


In [ ]:

def svd(A):
	# N is numeric eval
	A = Matrix(A)    # A is dxN
	rows, cols = A.shape
	Q = A*A.T  # Q is dxd
	pairs = []
	for eig, mult, vecs in Q.eigenvects():
		if N(eig) > 0:
			Ueig = gs(Matrix.hstack(*vecs))
			for j in range(mult):
				pairs.append((eig, Ueig[:, j]))
	pairs.sort(key = lambda p: N(-p[0]))
	U, sigma, V = zeros(rows, 0), zeros(0, 0), zeros(cols, 0)
	for eig, v in pairs:
		s = simplify(sqrt(eig))
		U = Matrix.hstack(U, simplify(A*v/s))
		V = Matrix.hstack(V, v)
		sigma = diag(sigma, s)
	return U, sigma, V


In [ ]:

from pandas import *
from numpy import *
from numpy.linalg import norm
from sklearn.decomposition import PCA

mnist = read_csv('mnist.csv').to_numpy()
dataset = mnist[:,1:]
labels = mnist[:,0]

engine = PCA(n_components = 4)
reduced = engine.fit_transform(dataset)      # shape (60000, 4)

dist = norm(reduced[:1000], axis = 1)        # distance to the origin of R^4
i = argmin(dist)
i, dist[i], labels[i]


In [ ]:

# EVD of the variance matrix
mu = mean(dataset, axis = 0)
Q = cov(dataset.T)
lamda, V = eigh(Q)
Vproj = V[:, ::-1][:, :4]                    # top 4 eigenvectors
reduced = dot(dataset - mu, Vproj)

# SVD of the d x N centered dataset matrix
A = (dataset - mu).T
U, sigma, Vt = svd(A)
reduced = dot(dataset - mu, U[:, :4])        # left singular vectors
